# Implement Layer Normalization from Scratch
### Problem Statement
Layer Normalization (LayerNorm), introduced in [Ba et al. 2016](https://arxiv.org/abs/1607.06450), normalizes activations across the **feature dimension** for each individual sample. Unlike Batch Normalization (which normalizes across the batch), LayerNorm is independent of batch size and works naturally for sequence models.

LayerNorm is used in the original Transformer (Vaswani et al. 2017), GPT-2, GPT-3, and BERT. Modern LLMs have largely replaced it with RMSNorm, but understanding LayerNorm is foundational.

Your goal is to implement LayerNorm from scratch and validate against `torch.nn.LayerNorm`.

---

### Requirements

1. **Compute Mean and Variance**
   - Mean: `mu = mean(x)` over the last dimension
   - Variance: `sigma² = var(x)` over the last dimension (unbiased=False)

2. **Normalize**
   - `x_norm = (x - mu) / sqrt(sigma² + eps)`

3. **Affine Transform**
   - Learnable `gamma` (scale, init ones) and `beta` (bias, init zeros)
   - `output = gamma * x_norm + beta`

4. **Validate Against PyTorch**
   - Compare with `torch.nn.LayerNorm` using `torch.allclose()`.

---

### Constraints

- ✅ Use only PyTorch operations.
- ✅ Normalize over the last dimension.
- ✅ Must match `torch.nn.LayerNorm` output.

---

<details>
  <summary>💡 Hint</summary>

  - Use `x.mean(dim=-1, keepdim=True)` and `x.var(dim=-1, keepdim=True, unbiased=False)`.
  - PyTorch's `LayerNorm` uses **population variance** (unbiased=False), not sample variance.
  - `gamma = nn.Parameter(torch.ones(dim))`, `beta = nn.Parameter(torch.zeros(dim))`.

</details>

---

In [ ]:
import torch
import torch.nn as nn

In [ ]:
class LayerNorm(nn.Module):
    """
    Implements Layer Normalization.
    """
    def __init__(self, dim: int, eps: float = 1e-5):
        super().__init__()
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(dim))   # scale
        self.beta = nn.Parameter(torch.zeros(dim))   # shift

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x (Tensor): Input tensor of shape (..., dim)

        Returns:
            Tensor: Normalized tensor of same shape
        """
        # Compute mean and variance over the last dimension
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)

        # Normalize
        x_norm = (x - mean) / torch.sqrt(var + self.eps)

        # Scale and shift
        return self.gamma * x_norm + self.beta

## 📚 LayerNorm: Key Concepts

### Why unbiased=False?

PyTorch's `nn.LayerNorm` uses **population variance** (divides by N, not N-1). This is because:
- We're normalizing a fixed set of features, not estimating a population parameter
- The feature dimension is the entire "population" for each sample

```python
# Population variance (unbiased=False): sum((x - mean)²) / N
# Sample variance (unbiased=True):      sum((x - mean)²) / (N-1)
x.var(dim=-1, unbiased=False)  # This matches nn.LayerNorm
```

### BatchNorm vs LayerNorm

```
Input shape: (batch_size, seq_len, dim)

BatchNorm:  normalizes across batch dimension     → stats shape: (dim,)
LayerNorm:  normalizes across feature dimension   → stats shape: (batch, seq_len, 1)
```

| Property | BatchNorm | LayerNorm |
|----------|-----------|----------|
| Normalizes over | Batch dim | Feature dim |
| Batch-size dependent? | Yes | No |
| Running stats at inference? | Yes | No |
| Common in | CNNs | Transformers |

### Pre-Norm vs Post-Norm

```
Post-Norm (original Transformer):     x + Sublayer(LayerNorm(x))  ← NO
Actually:                              LayerNorm(x + Sublayer(x))  ← YES

Pre-Norm (GPT-2, modern LLMs):        x + Sublayer(Norm(x))
```

Pre-Norm is more stable for training deep models because the residual path is unobstructed.

In [ ]:
# Validate against PyTorch's LayerNorm
torch.manual_seed(42)
x = torch.randn(3, 4, 8)  # (batch, seq_len, dim)

custom_ln = LayerNorm(dim=8)
pytorch_ln = nn.LayerNorm(8)

# Copy weights for fair comparison
custom_ln.gamma.data = pytorch_ln.weight.data.clone()
custom_ln.beta.data = pytorch_ln.bias.data.clone()

output_custom = custom_ln(x)
output_ref = pytorch_ln(x)

print("Custom output:")
print(output_custom[0, 0, :])
print("\nPyTorch output:")
print(output_ref[0, 0, :])

assert torch.allclose(output_custom, output_ref, atol=1e-6)
print("\n✅ Outputs match! LayerNorm implementation is correct.")